# Random Forest na Classificação do Desempenho dos Alunos

# Seção 1: Importação do Dataset e Dataset Usado

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [2]:
df = pd.read_parquet("dados_nota_enem.parquet")

# Seção 3: Achando os Melhores Parâmetros

In [3]:
n_estimators = [int(x) for x in np.linspace(start=50, stop=100, num=6)]
max_features = ['sqrt', 'log2']         
max_depth = [int(x) for x in np.linspace(start=10, stop=40, num=4)]
max_depth.append(None)

param_grid = {
    'n_estimators':      n_estimators,   
    'max_features':      max_features,   
    'max_depth':         max_depth,     
    'min_samples_split': [10, 20, 50],   
    'min_samples_leaf':  [10, 25, 50],   
}

In [4]:
df_reduzido = df.sample(n=100_000, random_state=42)

In [5]:
X = df_reduzido.drop(['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC','TP_SEXO', 'TP_COR_RACA',
       'NU_NOTA_MT', 'TX_RESPOSTAS_CN', 'TX_RESPOSTAS_CH', 'TX_RESPOSTAS_LC',
       'TX_RESPOSTAS_MT', 'TX_GABARITO_CN', 'TX_GABARITO_CH', 'TX_GABARITO_LC',
        'CLASSE_NOTA', 'MEDIA', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT',
       'TX_GABARITO_MT', 'TP_STATUS_REDACAO', 'NU_NOTA_COMP1', 'NU_NOTA_COMP2','IN_TREINEIRO', 
       'NU_NOTA_COMP3', 'NU_NOTA_COMP4', 'NU_NOTA_COMP5', 'NU_NOTA_REDACAO'], axis=1)

y = df_reduzido['CLASSE_NOTA']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
rf = RandomForestClassifier(class_weight='balanced', random_state=42)

cv_rf = RandomizedSearchCV(   
    estimator=rf,
    param_distributions=param_grid,
    n_iter=10,       
    cv=3,            
    scoring='f1_weighted',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

cv_rf.fit(x_train, y_train)

print(cv_rf.best_estimator_)
print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(x_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  cv_rf.predict(x_test))))
print(classification_report(y_test, cv_rf.predict(x_test)))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
RandomForestClassifier(class_weight='balanced', max_depth=30,
                       min_samples_leaf=10, min_samples_split=50,
                       n_estimators=60, random_state=42)
Ein:  0.2360
Eout: 0.2718
              precision    recall  f1-score   support

           0       0.75      0.68      0.71     10030
           1       0.71      0.78      0.74      9970

    accuracy                           0.73     20000
   macro avg       0.73      0.73      0.73     20000
weighted avg       0.73      0.73      0.73     20000

